# Smog City — Week 6: Feature Engineering

**Dataset:** `aqi_india_38cols_knn_final.csv`

## Aim
Prepare useful analysis features: parse dates, derive year/month/season where possible, encode categorical variables for modelling, and scale numeric features for comparable ranges.

This notebook keeps the original data unchanged and writes separate output files. Encoding and scaling are for analytical preparation; they do not prove a predictive model is accurate.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from google.colab import files
from IPython.display import display

pd.set_option("display.max_columns", 100)
print("Week 6 — Feature Engineering")

Week 6 — Feature Engineering


In [2]:
uploaded = files.upload()
filename = next(iter(uploaded))
df = pd.read_csv(filename, low_memory=False)
df.columns = df.columns.astype(str).str.strip().str.lower()

print("Loaded:", filename, "| shape:", df.shape)
display(df.head())

Saving aqi_india_38cols_knn_final.csv to aqi_india_38cols_knn_final.csv
Loaded: aqi_india_38cols_knn_final.csv | shape: (842160, 31)


,city,state,latitude,longitude,datetime,month,day_name,is_weekend,season,time_of_day,humidity_percent,dew_point_c,wind_gusts_kmh,precipitation_mm,is_raining,heavy_rain,pressure_msl_hpa,cloud_cover_percent,pm2_5_ugm3,pm10_ugm3,co_ugm3,no2_ugm3,so2_ugm3,o3_ugm3,dust_ugm3,aod,us_aqi,aqi_category,pm25_category_india,festival_period,crop_burning_season
0,agartala,tripura,23.8315,91.2868,2022-08-05 00:00:00,8.0,friday,False,monsoon,night,91.0,25.1,11.2,0.0,False,False,1004.4,100.0,14.8,21.5,197.0,21.8,2.7,32.0,0.0,0.14,289.0,Very Unhealthy,good,False,False
1,agartala,tripura,23.8315,91.2868,2022-08-05 01:00:00,8.0,friday,False,monsoon,night,93.0,25.2,9.0,0.0,False,False,1003.9,100.0,15.7,22.8,196.0,18.5,3.0,33.0,0.0,0.14,289.0,Very Unhealthy,good,False,False
2,agartala,tripura,23.8315,91.2868,2022-08-05 02:00:00,8.0,friday,False,monsoon,night,95.0,25.2,7.6,0.0,False,False,1003.5,100.0,16.3,23.5,196.0,15.1,3.3,34.0,0.0,0.15,54.0,Moderate,good,False,False
3,agartala,tripura,23.8315,91.2868,2022-08-05 03:00:00,8.0,friday,False,monsoon,night,96.0,25.2,6.8,0.0,False,False,1003.0,100.0,17.6,25.4,197.0,14.1,3.3,32.0,0.0,0.15,54.0,Moderate,good,False,False
4,agartala,tripura,23.8315,91.2868,2022-08-05 04:00:00,8.0,friday,False,monsoon,night,97.0,25.2,6.8,0.0,False,False,1002.9,100.0,18.2,26.2,199.0,13.9,3.2,30.0,0.0,0.14,54.0,Moderate,good,False,False


In [3]:
# 1. Parse datetime and derive time features when available
date_candidates = ["datetime", "date", "timestamp", "time"]
date_col = next((c for c in date_candidates if c in df.columns), None)

if date_col:
    parsed = pd.to_datetime(df[date_col], errors="coerce")
    df["derived_year"] = parsed.dt.year
    df["derived_month"] = parsed.dt.month
    df["derived_day"] = parsed.dt.day
    df["derived_hour"] = parsed.dt.hour
    print("Time features derived from:", date_col)
    print("Unparseable date values:", int(parsed.isna().sum()))
else:
    print("No recognizable date/time column found; skipping date-derived features.")

# Derive season only if there is a parseable date column
if date_col:
    month = pd.to_datetime(df[date_col], errors="coerce").dt.month
    season_map = {
        12: "Winter", 1: "Winter", 2: "Winter",
        3: "Summer", 4: "Summer", 5: "Summer",
        6: "Monsoon", 7: "Monsoon", 8: "Monsoon", 9: "Monsoon",
        10: "Post-Monsoon", 11: "Post-Monsoon"
    }
    df["derived_season"] = month.map(season_map)

display(df.head())

Time features derived from: datetime
Unparseable date values: 0


,city,state,latitude,longitude,datetime,month,day_name,is_weekend,season,time_of_day,humidity_percent,dew_point_c,wind_gusts_kmh,precipitation_mm,is_raining,heavy_rain,pressure_msl_hpa,cloud_cover_percent,pm2_5_ugm3,pm10_ugm3,co_ugm3,no2_ugm3,so2_ugm3,o3_ugm3,dust_ugm3,aod,us_aqi,aqi_category,pm25_category_india,festival_period,crop_burning_season,derived_year,derived_month,derived_day,derived_hour,derived_season
0,agartala,tripura,23.8315,91.2868,2022-08-05 00:00:00,8.0,friday,False,monsoon,night,91.0,25.1,11.2,0.0,False,False,1004.4,100.0,14.8,21.5,197.0,21.8,2.7,32.0,0.0,0.14,289.0,Very Unhealthy,good,False,False,2022,8,5,0,Monsoon
1,agartala,tripura,23.8315,91.2868,2022-08-05 01:00:00,8.0,friday,False,monsoon,night,93.0,25.2,9.0,0.0,False,False,1003.9,100.0,15.7,22.8,196.0,18.5,3.0,33.0,0.0,0.14,289.0,Very Unhealthy,good,False,False,2022,8,5,1,Monsoon
2,agartala,tripura,23.8315,91.2868,2022-08-05 02:00:00,8.0,friday,False,monsoon,night,95.0,25.2,7.6,0.0,False,False,1003.5,100.0,16.3,23.5,196.0,15.1,3.3,34.0,0.0,0.15,54.0,Moderate,good,False,False,2022,8,5,2,Monsoon
3,agartala,tripura,23.8315,91.2868,2022-08-05 03:00:00,8.0,friday,False,monsoon,night,96.0,25.2,6.8,0.0,False,False,1003.0,100.0,17.6,25.4,197.0,14.1,3.3,32.0,0.0,0.15,54.0,Moderate,good,False,False,2022,8,5,3,Monsoon
4,agartala,tripura,23.8315,91.2868,2022-08-05 04:00:00,8.0,friday,False,monsoon,night,97.0,25.2,6.8,0.0,False,False,1002.9,100.0,18.2,26.2,199.0,13.9,3.2,30.0,0.0,0.14,54.0,Moderate,good,False,False,2022,8,5,4,Monsoon


In [4]:
# 2. Choose a target/measurement set and inspect numeric distributions
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
print("Numeric columns found:", len(numeric_cols))
print(numeric_cols)

# Save numeric feature summary
if numeric_cols:
    display(df[numeric_cols].describe().T)
    df[numeric_cols].describe().T.to_csv("Week6_Numeric_Feature_Summary.csv")

Numeric columns found: 22
['latitude', 'longitude', 'month', 'humidity_percent', 'dew_point_c', 'wind_gusts_kmh', 'precipitation_mm', 'pressure_msl_hpa', 'cloud_cover_percent', 'pm2_5_ugm3', 'pm10_ugm3', 'co_ugm3', 'no2_ugm3', 'so2_ugm3', 'o3_ugm3', 'dust_ugm3', 'aod', 'us_aqi', 'derived_year', 'derived_month', 'derived_day', 'derived_hour']


,count,mean,std,min,25%,50%,75%,max
latitude,842160.0,23.129041,5.543478,8.5241,20.2961,23.8315,26.9124,31.1048
longitude,842160.0,82.769207,6.937205,72.5714,77.1734,80.9462,88.6065,94.1086
month,842160.0,6.801653,3.411504,1.0000,4.0000,7.0000,10.0000,12.0000
humidity_percent,842160.0,71.324796,21.362100,3.0000,58.0000,76.0000,89.0000,100.0000
dew_point_c,842160.0,17.160777,7.020427,-11.5000,12.1000,18.6000,23.3000,27.3000
wind_gusts_kmh,842160.0,19.270394,9.921461,3.2000,11.5000,17.3000,25.2000,55.1000
precipitation_mm,842160.0,0.197799,0.915802,0.0000,0.0000,0.0000,0.0000,51.7000
pressure_msl_hpa,842160.0,1009.489776,5.698795,979.4000,1005.4000,1009.9000,1013.8000,1028.1000
cloud_cover_percent,842160.0,50.501230,43.413480,0.0000,2.0000,47.0000,100.0000,100.0000
pm2_5_ugm3,842160.0,34.549825,29.652013,1.9000,14.7000,25.9000,44.2000,183.5000


In [5]:
# 3. One-hot encode selected categorical columns
# Limit to low-cardinality categoricals to avoid huge output files.
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
low_cardinality_cols = [c for c in categorical_cols if df[c].nunique(dropna=True) <= 30]

print("Categorical columns:", categorical_cols)
print("One-hot encoding these columns (<=30 unique values):", low_cardinality_cols)

# To keep the export manageable, use a sample if the full table is very large.
MAX_ROWS_FOR_ENCODED_EXPORT = 200000
encode_df = df.copy()
if len(encode_df) > MAX_ROWS_FOR_ENCODED_EXPORT:
    print(f"Dataset has {len(encode_df):,} rows; encoding a reproducible {MAX_ROWS_FOR_ENCODED_EXPORT:,}-row sample for demonstration.")
    encode_df = encode_df.sample(MAX_ROWS_FOR_ENCODED_EXPORT, random_state=42)

encoded_df = pd.get_dummies(
    encode_df,
    columns=low_cardinality_cols,
    dummy_na=False,
    dtype=int
)
print("Encoded shape:", encoded_df.shape)
display(encoded_df.head())
encoded_df.to_csv("Week6_Feature_Engineered_Encoded.csv", index=False)
print("Saved: Week6_Feature_Engineered_Encoded.csv")

Categorical columns: ['city', 'state', 'datetime', 'day_name', 'is_weekend', 'season', 'time_of_day', 'is_raining', 'heavy_rain', 'aqi_category', 'pm25_category_india', 'festival_period', 'crop_burning_season', 'derived_season']
One-hot encoding these columns (<=30 unique values): ['city', 'state', 'day_name', 'is_weekend', 'season', 'time_of_day', 'is_raining', 'heavy_rain', 'aqi_category', 'pm25_category_india', 'festival_period', 'crop_burning_season', 'derived_season']
Dataset has 842,160 rows; encoding a reproducible 200,000-row sample for demonstration.
Encoded shape: (200000, 123)


,latitude,longitude,datetime,month,humidity_percent,dew_point_c,wind_gusts_kmh,precipitation_mm,pressure_msl_hpa,cloud_cover_percent,pm2_5_ugm3,pm10_ugm3,co_ugm3,no2_ugm3,so2_ugm3,o3_ugm3,dust_ugm3,aod,us_aqi,derived_year,derived_month,derived_day,derived_hour,city_agartala,city_ahmedabad,city_aizawl,city_bengaluru,city_bhopal,city_bhubaneswar,city_chandigarh,city_chennai,city_dehradun,city_delhi,city_gangtok,city_gurugram,city_guwahati,city_hyderabad,city_imphal,city_itanagar,city_jaipur,city_kohima,city_kolkata,city_lucknow,city_mumbai,city_panaji,city_patna,city_raipur,city_ranchi,city_shillong,city_shimla,...,state_rajasthan,state_sikkim,state_tamil nadu,state_telangana,state_tripura,state_uttar pradesh,state_uttarakhand,state_west bengal,day_name_friday,day_name_monday,day_name_saturday,day_name_sunday,day_name_thursday,day_name_tuesday,day_name_wednesday,is_weekend_False,is_weekend_True,season_monsoon,season_post_monsoon,season_summer,season_winter,time_of_day_afternoon,time_of_day_early_morning,time_of_day_evening,time_of_day_morning,time_of_day_night,time_of_day_night_late,is_raining_False,is_raining_True,heavy_rain_False,heavy_rain_True,aqi_category_Good,aqi_category_Moderate,aqi_category_Unhealthy,aqi_category_Unhealthy for Sensitive Groups,aqi_category_Very Unhealthy,pm25_category_india_good,pm25_category_india_moderate,pm25_category_india_poor,pm25_category_india_satisfactory,pm25_category_india_severe,pm25_category_india_very_poor,festival_period_False,festival_period_True,crop_burning_season_False,crop_burning_season_True,derived_season_Monsoon,derived_season_Post-Monsoon,derived_season_Summer,derived_season_Winter
160063,20.2961,85.8245,2024-04-15 07:00:00,4.0,61.0,21.0,7.9,0.0,1011.6,0.0,37.3,58.1,408.0,5.4,9.5,91.0,2.0,0.63,111.0,2024,4,15,7,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,1,0,0,1,0,1,0,0,0,1,0
541575,22.5726,88.3639,2024-09-28 15:00:00,9.0,74.0,25.4,33.5,0.2,1005.7,95.0,8.2,13.1,164.0,5.1,4.6,62.0,0.0,0.24,40.0,2024,9,28,15,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,1,1,0,1,0,0,0,0,1,0,0,0,0,0,1,0,1,0,1,0,0,0
322601,28.4595,77.0266,2022-12-14 17:00:00,12.0,48.0,5.9,18.4,0.0,1015.1,0.0,1.9,2.3,96.0,0.0,0.8,27.0,0.0,0.05,19.0,2022,12,14,17,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,1,1,0,0,0,0,0,1,0,1,0,1,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,1
794503,8.5241,76.9366,2023-10-13 07:00:00,10.0,93.0,24.0,9.4,0.0,1013.1,5.0,8.5,13.3,194.0,6.7,3.2,46.0,0.0,0.43,24.0,2023,10,13,7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,1,0,1,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,1,0,0
483702,26.9124,75.7873,2024-10-07 06:00:00,10.0,80.0,18.8,10.4,0.0,1014.3,0.0,26.6,51.0,421.0,22.6,8.8,37.0,32.0,0.25,78.0,2024,10,7,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,1,0,1,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,1,0,1,0,0


Saved: Week6_Feature_Engineered_Encoded.csv


In [6]:
# 4. Standardize numeric columns in the working data
# Exclude derived calendar parts only if you do not want them in scaling? Here all numeric columns are scaled.
scale_cols = df.select_dtypes(include=np.number).columns.tolist()

if scale_cols:
    # Use complete numeric rows for a clear scaler demonstration.
    scale_source = df[scale_cols].replace([np.inf, -np.inf], np.nan)
    complete_rows = scale_source.dropna()
    if len(complete_rows) > 0:
        scaler = StandardScaler()
        scaled_array = scaler.fit_transform(complete_rows)
        scaled_df = pd.DataFrame(scaled_array, columns=scale_cols, index=complete_rows.index)
        print("Rows scaled:", len(scaled_df), "Numeric columns:", len(scale_cols))
        display(scaled_df.head())
        scaled_df.to_csv("Week6_Scaled_Numeric_Features.csv", index=False)
        print("Saved: Week6_Scaled_Numeric_Features.csv")
    else:
        print("No complete numeric rows available to scale.")
else:
    print("No numeric columns found; scaling skipped.")

Rows scaled: 842160 Numeric columns: 22


,latitude,longitude,month,humidity_percent,dew_point_c,wind_gusts_kmh,precipitation_mm,pressure_msl_hpa,cloud_cover_percent,pm2_5_ugm3,pm10_ugm3,co_ugm3,no2_ugm3,so2_ugm3,o3_ugm3,dust_ugm3,aod,us_aqi,derived_year,derived_month,derived_day,derived_hour
0,0.126718,1.227814,0.351267,0.921034,1.130875,-0.813428,-0.215984,-0.893132,1.140171,-0.666054,-0.599554,-0.657634,0.291925,-0.564982,-1.076541,-0.274412,-1.080187,4.113241,-1.73243,0.351267,-1.224634,-1.661325
1,0.126718,1.227814,0.351267,1.014658,1.145119,-1.035170,-0.215984,-0.980870,1.140171,-0.635702,-0.575583,-0.660287,0.126064,-0.549127,-1.054201,-0.274412,-1.080187,4.113241,-1.73243,0.351267,-1.224634,-1.516862
2,0.126718,1.227814,0.351267,1.108281,1.145119,-1.176278,-0.215984,-1.051061,1.140171,-0.615467,-0.562675,-0.660287,-0.044822,-0.533272,-1.031861,-0.274412,-1.046841,-0.861912,-1.73243,0.351267,-1.224634,-1.372399
3,0.126718,1.227814,0.351267,1.155093,1.145119,-1.256912,-0.215984,-1.138798,1.140171,-0.571625,-0.527640,-0.657634,-0.095082,-0.533272,-1.076541,-0.274412,-1.046841,-0.861912,-1.73243,0.351267,-1.224634,-1.227936
4,0.126718,1.227814,0.351267,1.201905,1.145119,-1.256912,-0.215984,-1.156346,1.140171,-0.551390,-0.512888,-0.652329,-0.105134,-0.538557,-1.121222,-0.274412,-1.080187,-0.861912,-1.73243,0.351267,-1.224634,-1.083473


Saved: Week6_Scaled_Numeric_Features.csv


In [7]:
# 5. Compare original and scaled numeric ranges for the first few columns
if scale_cols and 'scaled_df' in globals() and len(scaled_df):
    compare_cols = scale_cols[:min(5, len(scale_cols))]
    comparison = pd.DataFrame({
        "original_mean": df.loc[scaled_df.index, compare_cols].mean(),
        "scaled_mean": scaled_df[compare_cols].mean(),
        "original_std": df.loc[scaled_df.index, compare_cols].std(),
        "scaled_std": scaled_df[compare_cols].std()
    })
    display(comparison.round(3))
    comparison.to_csv("Week6_Scaling_Comparison.csv")
    print("Saved: Week6_Scaling_Comparison.csv")

,original_mean,scaled_mean,original_std,scaled_std
latitude,23.129,-0.0,5.543,1.0
longitude,82.769,-0.0,6.937,1.0
month,6.802,0.0,3.412,1.0
humidity_percent,71.325,-0.0,21.362,1.0
dew_point_c,17.161,0.0,7.020,1.0


Saved: Week6_Scaling_Comparison.csv


## Week 6 conclusion
Record which time features were created, which categorical columns were encoded, and how scaling changed numeric feature ranges. Note whether encoding was run on the full dataset or on a sample, based on the output.